# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. This dataset contains ordered logistic regression outputs and survey results relating to rangeland management in Northern Kenya.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', '<no name>')}\n\nDescription: {getattr(metadata, 'description', '<no description>')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record set @id and their field @ids and column @ids
from pprint import pprint

record_sets = list(dataset.record_sets)

if len(record_sets) == 0:
    print('No record sets found in metadata.')
else:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"- Record Set @id: {rs['@id']}")
        fields = rs.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        print('  Fields:')
        for field in fields:
            if isinstance(field, dict):
                print(f"    - {field.get('@id', field)}")
            else:
                print(f"    - {field}")
        columns = rs.get('column', [])
        if not isinstance(columns, list):
            columns = [columns]
        print('  Columns:')
        for column in columns:
            if isinstance(column, dict):
                print(f"    - {column.get('@id', column)}")
            else:
                print(f"    - {column}")

Let's also display a preview of the first record in the first record set.

In [ ]:
# Loop over records in the first available record set
if len(record_sets) > 0:
    main_rs_id = record_sets[0]['@id']
    print(f"\nSampling a record from Record Set {main_rs_id}:\n")
    try:
        for rec in dataset.records(record_set=main_rs_id):
            pprint(rec)
            break
    except Exception as e:
        print(f"Failed to load records from record set {main_rs_id}: {e}")
else:
    print("No record sets to preview records from.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        dataframes[rsid] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[rsid])} rows for record set {rsid}")
    except Exception as e:
        print(f"Could not load records for record set {rsid}: {e}")

# Display columns for the first record set (if exists)
if record_set_ids:
    first_rs_id = record_set_ids[0]
    if first_rs_id in dataframes:
        print(f"\nColumns in {first_rs_id}:")
        print(list(dataframes[first_rs_id].columns))
        display(dataframes[first_rs_id].head())
else:
    print("No record sets available to extract data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, select a record set and numeric field to analyze
if record_set_ids:
    rsid = record_set_ids[0]
    df = dataframes.get(rsid)
    if df is not None and not df.empty:
        print(f"Analyzing record set {rsid}, sample columns: {df.columns.tolist()}")
        
        # Try to auto-select a numeric field
        numeric_field_id = None
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        if numeric_field_id is None:
            print('No numeric field found for demonstration.')
        else:
            threshold = df[numeric_field_id].mean()  # Use mean as threshold example
            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
            print(filtered_df[[numeric_field_id]].head())
            filtered_df[f"{numeric_field_id}_normalized"] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            )
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Try grouping by the first string/categorical field
            group_field = None
            for col in df.columns:
                if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
                    group_field = col
                    break
            if group_field:
                grouped_df = (
                    filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
                )
                print(f"\nAverage {numeric_field_id} by {group_field}:")
                print(grouped_df.head())
            else:
                print("No categorical field available for grouping.")
    else:
        print('No data available for EDA in the selected record set.')
else:
    print("No record sets loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple distribution plot for the selected numeric field
if record_set_ids and df is not None and numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If there was a group_field, plot boxplot
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step loading, exploration, and simple analysis of the FAIR² dataset using `mlcroissant`. We:

- Loaded and examined the dataset metadata.
- Listed record sets, fields, and columns using their Croissant `@id`s.
- Loaded dataframes for available record sets, previewed records, and performed basic EDA.
- Visualized a selected numeric field and group distribution (if applicable).

For further analysis, review the Croissant schema and documentation for additional record sets, fields, or relationships within the data.
